# Generating Stochastic Volatility Surface Datasets

Author: Sebastien Gurrieri, sebgur@gmail.com

This notebook generates datasets for the future training of models to fit implied volatility surfaces, see [training workbook](https://colab.research.google.com/drive/1y-Tb4JxiBmcJAw943GWtZn_GiGhcUUzS#scrollTo=RGQNx52-lrZ5) for more details about the models and references to the literature. For all supported models, we generate samples defined by:

* expiry, strike, forward rate
* lognormal vol level: order of magnitude in lognormal terms for easy intuition. This will be transformed into a model parameter in a model-dependent way. For instance it will be transformed into the parameter $\alpha$ for SABR.
* other model parameters (say $\beta, \nu, rho$ for SABR)
* option price (put): calculated with the closed-form for Hagan SABR, or by Monte-Carlo simulation for the other models i.e. No-Arbitrage SABR, Free-Boundary SABR, ZABR and Heston.

We start by randomly generating expiry, strike, forward rate and other parameters over a user-specified range. For each sample we then calculate the option price. Finally we go through a process of data cleansing which, at the moment, simply consists in calculating the normal volatilities and rejecting numbers judged too high and too low. We then output the cleansed data to a tsv file.

As the quality of the data is crucial for the subsequent training, it would be interesting to consider more sophisticated approaches for filtering out bad data, together with of course more efficient valuation methods such as PDEs. The current version of this code, running on a 2018 i5 CPU, takes about 10 hours to generate 100k samples for MC based models.

In [2]:
# Import relevant modules and set project folder
# import os
from pathlib import Path
from datetime import datetime
from sdevpy.utilities.timer import Stopwatch
from sdevpy.volatility.mlsurfacegen import stovolfactory


project_path = Path.home() / "temp" / "sdevpy" / "stovol"
project_path.mkdir(parents=True, exist_ok=True)
print(f"Project folder: {project_path}")

Project folder: /Users/sebgur/temp/sdevpy/stovol


## 1) Set runtime configuration


In [10]:
# Global settings
MODEL_TYPE = "SABR"  # SABR, McSABR, FbSABR, McZABR, McHeston
SHIFT = 0.03
NUM_SAMPLES = 100_000
# The 4 parameters below are only relevant for models whose reference is calculated by MC
NUM_EXPIRIES = 10
NUM_STRIKES = 5
NUM_MC = 10 * 1000 # 100 * 1000
POINTS_PER_YEAR = 25 # 25
# Change seed to generate different sets
SEED = 42  # [2468, 8642, 2112, 4444, 88, 6666, 1122, 12]

dataset_path = project_path / "datasets"
print(f"Data folder: {dataset_path}")
dataset_path.mkdir(parents=True, exist_ok=True)
print(f"Chosen model: {MODEL_TYPE}")
data_file = dataset_path / f"{MODEL_TYPE}_data.tsv"

Data folder: /Users/sebgur/temp/sdevpy/stovol/datasets
Chosen model: SABR


## 2) Generate samples

Here we generate the samples using the SDevPy framework. First prices are calculated with the chosen models. Then these prices are transformed into normal volatilities and the data is cleansed. Finally a tsv file is output containing the dataset.

In [11]:
# Select the model
generator = stovolfactory.set_generator(MODEL_TYPE, SHIFT, NUM_EXPIRIES, NUM_STRIKES, NUM_MC,
                                        POINTS_PER_YEAR, SEED)

In [12]:
# Select training range
# SABR
RANGES = {'Ttm': [1.0 / 12.0, 35.0], 'K': [0.01, 0.99], 'F': [-0.009, 0.041], 'LnVol': [0.05, 0.5],
          'Beta': [0.1, 0.9], 'Nu': [0.1, 1.0], 'Rho': [-0.6, 0.6]}
# # FBSABR
# RANGES = {'Ttm': [1.0 / 12.0, 10.0], 'K': [0.01, 0.99], 'F': [-0.01, 0.05], 'LnVol': [0.05, 0.5],
#           'Beta': [0.25, 0.75], 'Nu': [0.1, 1.0], 'Rho': [-0.6, 0.6]}
# # ZABR
# RANGES = {'Ttm': [1.0 / 12.0, 35.0], 'K': [0.01, 0.99], 'F': [-0.009, 0.041], 'LnVol': [0.05, 0.5],
#           'Beta': [0.1, 0.9], 'Nu': [0.10, 1.0], 'Rho': [-0.6, 0.6],
#           'Gamma': [0.1, 0.9]}
# Heston
# RANGES = {'Ttm': [1.0 / 12.0, 35.0], 'K': [0.01, 0.99], 'F': [-0.009, 0.041], 'LnVol': [0.05, 0.25],
#           'Kappa': [0.25, 4.00], 'Theta': [0.05**2, 0.25**2], 'Xi': [0.10, 0.50],
#           'Rho': [-0.40, 0.40]}

In [13]:
# Generate samples (prices)
print(f"Generate {NUM_SAMPLES:,} price samples")
timer_gen = Stopwatch("Generating Samples")
timer_gen.trigger()
data_df = generator.generate_samples(NUM_SAMPLES, RANGES)
timer_gen.stop()

# Convert to normal vols and cleanse
print("Convert to normal vol and cleanse data")
timer_conv = Stopwatch("Converting Prices")
timer_conv.trigger()
data_df = generator.to_nvol(data_df, cleanse=True)
num_clean = len(data_df.index)
print(f"Dataset size after cleansing: {num_clean:,}")
timer_conv.stop()

# Output to file
timer_out = Stopwatch("File Output")
timer_out.trigger()
now = datetime.now()
dt_string = now.strftime("%Y%m%d-%H_%M_%S")
data_file = dataset_path / f"{MODEL_TYPE}_data_{dt_string}.tsv"
print(f"Output to file {data_file}")
generator.to_file(data_df, data_file)
timer_out.stop()

# View timers
timer_gen.print()
timer_conv.print()
timer_out.print()

Generate 100,000 price samples
Convert to normal vol and cleanse data
Dataset size after cleansing: 99,877
Output to file /Users/sebgur/temp/sdevpy/stovol/datasets/SABR_data_20261121-13_59_16.tsv
Runtime(Generating Samples): 1.2s
Runtime(Converting Prices): 3.2s
Runtime(File Output): 0.3s
